Method: https://proghead231.github.io/agricultural-land-abandonment-nepal/
1. Data acquisition and preprocessing
2. Create agricultural land (AL) objects
3. Calculate annual AL probability
4. Apply landTrendr to the time series of AL probabilities
5. Identify structural breaks and classify eac AL objects to abandoned (AAL), fallow (FAL) or recultivate (RAL)
6. Validate results

## Step 1: Create AL objects
Given the uncertainity in the AL classification of the land cover datasets by FRTC I will do segmentation using some algo
1. Get landsat-SR images from 2000, 2010 and 2020, preprocess them and stack them 
2. Get texture metrics from the one image, combine the metrics into one texture image and add to the stack (Using GLCM metrics here)
3. Perform segmentation and check

In [1]:
import ee
# ee.Authenticate()
ee.Initialize(project="ee-joshisur231")
import geemap
import config
import utils
Map = geemap.Map()

loaded config!
loaded utils!


In [2]:
dem = ee.Image("USGS/SRTMGL1_003").clip(config.ROI).rename("z")
terrain = ee.Terrain.products(dem)

l_seg_bands= ee.List(["nir", "swir1", "swir2", "nir_savg", "nir_shade"])
terrain_bands = ee.List(["slope", "northness", "eastness"])

l7_2000_image = utils.get_processed_landsat_collection("LANDSAT/LE07/C02/T1_L2", config.ROI, config.LANDSAT_DATES["2000"], utils.mask_clouds_landsat75, utils.apply_scale_factors)\
    .median().clip(config.ROI)\
    .select(config.L75_ORIGINAL_BAND_NAMES).rename(config.L75_NEW_BAND_NAMES)\
    .select(config.L_FINAL_BANDS)

high_val_2000 = ee.Number(utils.calc_image_stats(l7_2000_image, config.ROI, config.SCALE).get("high"))
l7_2000_image = utils.add_scaled_glcm(l7_2000_image, config.ROI, config.SCALE, high_val_2000).select(l_seg_bands).rename(l_seg_bands.map(lambda band_name: ee.String(band_name).cat("_2000")))
terrain_2000 = utils.prepare_terrain_seg(terrain, config.ROI, scale = config.SCALE, high_val=high_val_2000).rename(terrain_bands.map(lambda band_name: ee.String(band_name).cat("_2000")))

l5_2010_image = utils.get_processed_landsat_collection("LANDSAT/LT05/C02/T1_L2", config.ROI, config.LANDSAT_DATES["2010"], utils.mask_clouds_landsat75, utils.apply_scale_factors)\
    .median().clip(config.ROI)\
    .select(config.L75_ORIGINAL_BAND_NAMES).rename(config.L75_NEW_BAND_NAMES)\
    .select(config.L_FINAL_BANDS)
high_val_2010 = ee.Number(utils.calc_image_stats(l5_2010_image, config.ROI, config.SCALE).get("high"))
l5_2010_image = utils.add_scaled_glcm(l5_2010_image, config.ROI, config.SCALE, high_val_2010).select(l_seg_bands).rename(l_seg_bands.map(lambda band_name: ee.String(band_name).cat("_2010")))
terrain_2010 = utils.prepare_terrain_seg(terrain, config.ROI, scale = config.SCALE, high_val=high_val_2010).rename(terrain_bands.map(lambda band_name: ee.String(band_name).cat("_2010")))

l8_2020_image = utils.get_processed_landsat_collection("LANDSAT/LC08/C02/T1_L2", config.ROI, config.LANDSAT_DATES["2020"], utils.mask_clouds_landsat8, utils.apply_scale_factors)\
    .median().clip(config.ROI)\
    .select(config.L8_ORIGINAL_BAND_NAMES).rename(config.L8_NEW_BAND_NAMES)\
    .select(config.L_FINAL_BANDS)
high_val_2020 = ee.Number(utils.calc_image_stats(l8_2020_image, config.ROI, config.SCALE).get("high"))
l8_2020_image = utils.add_scaled_glcm(l8_2020_image, config.ROI, config.SCALE, high_val_2020).select(l_seg_bands).rename(l_seg_bands.map(lambda band_name: ee.String(band_name).cat("_2020")))
terrain_2020 = utils.prepare_terrain_seg(terrain, config.ROI, scale = config.SCALE, high_val=high_val_2020).rename(terrain_bands.map(lambda band_name: ee.String(band_name).cat("_2020")))

seg_image_stack = l7_2000_image.addBands(l5_2010_image).addBands(l8_2020_image).addBands(terrain_2000).addBands(terrain_2010).addBands(terrain_2020)


In [3]:
# seg_image_stack.reproject(crs="EPSG:32645", scale=30).reduceRegion(ee.Reducer.minMax(), config.ROI, 30, maxPixels=21308891)

In [4]:
target_crs = "EPSG:32645"
target_scale = 30

seeds = ee.Algorithms.Image.Segmentation.seedGrid(15).reproject(crs=target_crs, scale=target_scale)
snic = ee.Algorithms.Image.Segmentation.SNIC(
  image= seg_image_stack.reproject(crs=target_crs, scale=target_scale),
  connectivity= 4,
  neighborhoodSize= 30,
  seeds= seeds
)
contours = snic.select('clusters') \
    .reduceToVectors(geometry=config.ROI, scale=30, maxPixels=1e9)

In [5]:
def get_s2_reference(roi, date_range):
    # 1. Define Sentinel-2 Cloud Mask
    def mask_s2_clouds(image):
        qa = image.select('QA60')
        # Bits 10 and 11 are clouds and cirrus, respectively.
        cloudBitMask = 1 << 10
        cirrusBitMask = 1 << 11
        mask = qa.bitwiseAnd(cloudBitMask).eq(0) \
            .And(qa.bitwiseAnd(cirrusBitMask).eq(0))
        return image.updateMask(mask).divide(10000) # Apply scaling (0.0001) here

    # 2. Fetch Collection
    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterDate(date_range[0], date_range[1]) \
        .filterBounds(roi) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
        .map(mask_s2_clouds) \
        .median() \
        .clip(roi)

    return s2

s2_dates = ["2020-01-01", "2021-12-28"]

# Get the image
s2_2020_image = get_s2_reference(config.ROI, s2_dates)

# Visualization Parameters (False Color is best for boundaries)
# B8 = NIR, B4 = Red, B3 = Green
vis_params_s2 = {
    'min': 0.0,
    'max': 0.3,
    'bands': ['B4', 'B3', 'B2'],
}
Map.addLayer(s2_2020_image, vis_params_s2, 'Sentinel-2 (10m) Ref')

# vis_params = {"min":0, "max":0.3, "bands": ["r", "g", "b"]} 
# Map.addLayer(ee.Image.constant(-9999).clip(config.ROI), {"palette":["red"]}, "bg")
# Map.addLayer(l7_2000_rgb, vis_params, "l7")
# Map.addLayer(seg_image_stack, {"min":0, "max": 1}, "final_stack")
Map.addLayer(
    contours.style(color='ffffff', width=1, fillColor='00000000'), 
    {}, 
    'Segment Boundaries'
)
# Map.addLayer(snic.select("clusters"), {}, "seg")
# Map.addLayer(l5_2010_image, vis_params, "l5")
# Map.addLayer(l8_2020_image, vis_params, "l8")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…